### Realizamos los imports que requeriremos para el analisis

In [70]:
import pandas as pd
import numpy as np

# Set de opciones de pandas.
pd.set_option('display.max_columns', None)

# 01 Estudio del dataset

In [71]:
df = pd.read_csv('../Datos/datosSpotify.csv', index_col=0)  
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 1.1 Dimensiones y estructura del dataset

El dataset contiene **114.000 registros y 20 variables**. Sin embargo, se han identificado únicamente **89.741 valores únicos de `track_id`**, lo que indica que existen canciones representadas en varias filas.

Las variables incluyen información de identificación de las canciones, popularidad, género musical y características de audio como energía, bailabilidad, acústica y valencia.

In [72]:
# Canciones unicas en el dataset
print(f"Filas totales: {df.shape[0]}")
print(f"Canciones únicas (track_id): {df['track_id'].nunique()}")

# Cuantas variables contiene el dataset
print(f"Número de variables: {df.shape[1]}")
print(df.columns.tolist())

Filas totales: 114000
Canciones únicas (track_id): 89741
Número de variables: 20
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']



## 1.2 Tipología de las variables

Se identifican tres grandes grupos de variables:

- **Identificativas y categóricas:** identificadores, artistas, álbumes, nombres de canciones y géneros.
- **Numéricas:** popularidad, duración y características musicales.
- **Booleanas:** indicador de contenido explícito.



In [73]:
# Qué tipo de datos tenemos
df.info()
# Resumen agrupado por tipo
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude=['number', 'bool']).columns.tolist()
bool_cols = df.select_dtypes(include='bool').columns.tolist()

print("Numéricas:", num_cols)
print("Texto/categóricas:", cat_cols)
print("Booleanas:", bool_cols)

# Resumen estadistico
df.describe().T

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  str    
 1   artists           113999 non-null  str    
 2   album_name        113999 non-null  str    
 3   track_name        113999 non-null  str    
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           114000 non-nu

,count,mean,std,min,25%,50%,75%,max
popularity,114000.0,33.238535,22.305078,0.000,17.00000,35.000000,50.0000,100.000
duration_ms,114000.0,228029.153114,107297.712645,0.000,174066.00000,212906.000000,261506.0000,5237295.000
danceability,114000.0,0.566800,0.173542,0.000,0.45600,0.580000,0.6950,0.985
energy,114000.0,0.641383,0.251529,0.000,0.47200,0.685000,0.8540,1.000
key,114000.0,5.309140,3.559987,0.000,2.00000,5.000000,8.0000,11.000
loudness,114000.0,-8.258960,5.029337,-49.531,-10.01300,-7.004000,-5.0030,4.532
mode,114000.0,0.637553,0.480709,0.000,0.00000,1.000000,1.0000,1.000
speechiness,114000.0,0.084652,0.105732,0.000,0.03590,0.048900,0.0845,0.965
acousticness,114000.0,0.314910,0.332523,0.000,0.01690,0.169000,0.5980,0.996
instrumentalness,114000.0,0.156050,0.309555,0.000,0.00000,0.000042,0.0490,1.000


## 1.3 Calidad inicial de los datos, duplicados

También se han identificado **450 filas completamente duplicadas**.

Además, existen **24.259 registros adicionales que comparten un `track_id` con una fila anterior**. Esta cifra no representa canciones únicas duplicadas, sino apariciones adicionales de identificadores existentes.

Por tanto, será necesario diferenciar los duplicados exactos de las canciones que aparecen asociadas a varios géneros.


In [74]:
# Duplicados
print("Filas idénticas:", df.duplicated().sum())
print("track_id repetidos:", df.duplicated(subset='track_id').sum())
print("Repetidas ignorando el género:", df.drop(columns='track_genre').duplicated().sum())

# Ejemplo de una canción en varios géneros
id_rep = df[df.duplicated('track_id', keep=False)]['track_id'].iloc[0]
display(df[df['track_id'] == id_rep][['track_name', 'artists', 'track_genre']])

Filas idénticas: 450
track_id repetidos: 24259
Repetidas ignorando el género: 23539


,track_name,artists,track_genre
0,Comedy,Gen Hoshino,acoustic
62102,Comedy,Gen Hoshino,j-pop
99152,Comedy,Gen Hoshino,singer-songwriter
102151,Comedy,Gen Hoshino,songwriter


### 1.3.1 Análisis de frecuencia y géneros

De los 89.741 identificadores únicos, **16.641 aparecen en varias filas**. La frecuencia máxima observada es de nueve apariciones por canción.

Por otro lado, **16.299 canciones están asociadas a más de un género**, con un máximo de nueve géneros diferentes.

Esto indica que el género no funciona como una clasificación exclusiva y que existe una relación de muchos a muchos entre canciones y categorías musicales.

Este comportamiento deberá tenerse en cuenta para evitar contar varias veces una misma canción en los análisis generales.

In [75]:
# Número de apariciones de cada canción
frecuencia = df['track_id'].value_counts()

# Estadísticas de frecuencia
print("Frecuencia media:", frecuencia.mean())
print("Máximo de apariciones:", frecuencia.max())

# Canciones que aparecen más de una vez
print("IDs que aparecen varias veces:", (frecuencia > 1).sum())

# Distribución de apariciones
display(frecuencia.value_counts().sort_index())

Frecuencia media: 1.2703223721598824
Máximo de apariciones: 9
IDs que aparecen varias veces: 16641


count
1    73100
2    11712
3     2984
4     1372
5      431
6      117
7       22
8        2
9        1
Name: count, dtype: int64

In [76]:
# Número de géneros diferentes por canción
generos_por_cancion = df.groupby('track_id')['track_genre'].nunique()

print("Media de géneros por canción:", generos_por_cancion.mean())
print("Máximo de géneros por canción:", generos_por_cancion.max())

display(generos_por_cancion.value_counts().sort_index())

Media de géneros por canción: 1.2653079417434618
Máximo de géneros por canción: 9


track_genre
1    73442
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: count, dtype: int64

In [77]:
repetidas = (
    df.groupby('track_id')
    .agg(
        nombre=('track_name', 'first'),
        artista=('artists', 'first'),
        apariciones=('track_id', 'size'),
        generos=('track_genre', lambda x: ', '.join(sorted(x.unique()))),
        num_generos=('track_genre', 'nunique'),
        popularidad_distinta=('popularity', 'nunique')
    )
    .query('apariciones > 1')
    .sort_values('apariciones', ascending=False)
)

display(repetidas.head(15))

,nombre,artista,apariciones,generos,num_generos,popularidad_distinta
track_id,,,,,,
6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,9,"blues, country, folk, j-pop, j-rock, power-pop...",9,1
2Ey6v4Sekh3Z0RUSISRosD,Layla,Derek & The Dominos,8,"blues, british, country, folk, hard-rock, psyc...",8,1
2kkvB3RNRzwjFdGhaUA0tz,Layla,Derek & The Dominos,8,"blues, british, country, folk, hard-rock, psyc...",8,1
5ZsAhuQ24mWHiduaxJqnhW,Udd Gaye,Ritviz,7,"edm, hip-hop, indian, indie, indie-pop, pop, p...",7,1
5ftfVzSLIi5ZxYdNbRtf41,Never Gonna Give You Up,The Black Keys,7,"alt-rock, alternative, blues, garage, punk, pu...",7,1
0RSGPiykniIg8m7JhiAVv7,Aisi Raaton,Anupam Roy,7,"indian, indie, indie-pop, k-pop, pop-film, sin...",7,1
61202Zzk9rk4bPMZLh2gr6,Trouble No More,Allman Brothers Band,7,"blues, country, folk, hard-rock, j-rock, singe...",7,1
7tbzfR8ZvZzJEzy6v0d6el,Liggi,Ritviz,7,"edm, hip-hop, indian, indie, indie-pop, pop, p...",7,1
2vU6bm5hVF2idVknGzqyPL,Christmas (Baby Please Come Home),The Offspring,7,"alt-rock, alternative, metal, punk, punk-rock,...",7,1


In [78]:
# Filas con track_id repetido que no son duplicadas
# al ignorar el género
mask = df.duplicated('track_id', keep=False)

grupo = df.loc[mask].copy()

no_duplicadas = grupo.loc[
    ~grupo.drop(columns='track_genre').duplicated(keep=False)
]

print("Registros a investigar:", len(no_duplicadas))

display(
    no_duplicadas.sort_values('track_id').head(20)
)

Registros a investigar: 1058


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
91138,014SIjoLDG1Ku19c5FlDYh,Creedence Clearwater Revival,Pumpkin Patch Hits,I Put A Spell On You,3,271786,False,0.393,0.732,4,-8.188,0,0.0295,0.004710,0.123000,0.0979,0.621,100.410,4,rock
19042,014SIjoLDG1Ku19c5FlDYh,Creedence Clearwater Revival,Pumpkin Patch Hits,I Put A Spell On You,0,271786,False,0.393,0.732,4,-8.188,0,0.0295,0.004710,0.123000,0.0979,0.621,100.410,4,country
68773,02jLfqc9gMo8PkHEGHY3OT,Feid,Si Te La Encuentras Por Ahí,Si Te La Encuentras Por Ahí,83,191573,False,0.743,0.576,8,-3.552,0,0.0760,0.151000,0.000059,0.1280,0.455,171.945,4,latino
91912,03mHinvLdrdSTd7w4GPXwH,Arctic Monkeys,I Ain't Quite Where I Think I Am,There’d Better Be A Mirrorball,58,265798,False,0.511,0.367,3,-10.891,1,0.0319,0.642000,0.012200,0.1860,0.323,143.698,4,rock
109084,04IUJPcxiUuXih8eJIVzGz,Tube & Berger;Mat.Joe;Tex Amos,"Best of Kittball, Vol. 6",Now Is Not Forever,2,234330,False,0.667,0.925,11,-7.522,0,0.0547,0.001970,0.136000,0.0630,0.792,126.976,4,techno
81529,04V3hO138BCddFy9TmTUic,Feid;Sech,Halloween 2022 Perreo Vol. 4,JAMAICA,5,224240,False,0.520,0.542,1,-7.404,0,0.1550,0.038300,0.000000,0.3710,0.669,81.131,4,pop
98764,04VFfRka9MBc5qlRcDuV8T,Alan Menken;Disney,A Pequena Sereia (Trilha Sonora Original em Po...,Fanfarra,21,30080,False,0.599,0.280,2,-14.787,1,0.0843,0.796000,0.007510,0.0634,0.564,119.539,4,show-tunes
26672,04VFfRka9MBc5qlRcDuV8T,Alan Menken;Disney,A Pequena Sereia (Trilha Sonora Original em Po...,Fanfarra,22,30080,False,0.599,0.280,2,-14.787,1,0.0843,0.796000,0.007510,0.0634,0.564,119.539,4,disney
107394,05UkW0aWSlpxcOlT3NtJ1N,Grace Jones,Classic Grace Jones,I've Seen That Face Before (Libertango),29,268800,False,0.807,0.460,9,-14.432,0,0.0564,0.000619,0.111000,0.0604,0.965,133.329,4,synth-pop
25934,05UkW0aWSlpxcOlT3NtJ1N,Grace Jones,Classic Grace Jones,I've Seen That Face Before (Libertango),30,268800,False,0.807,0.460,9,-14.432,0,0.0564,0.000619,0.111000,0.0604,0.965,133.329,4,disco


In [79]:
# Identificamos los track_id que aparecen más de una vez
ids_repetidos = df['track_id'].value_counts()
ids_repetidos = ids_repetidos[ids_repetidos > 1].index

# Seleccionamos los registros repetidos
df_repetidas = df[df['track_id'].isin(ids_repetidos)].copy()

# Calculamos cuántos valores diferentes tiene cada variable
# para cada canción
variaciones = (
    df_repetidas
    .groupby('track_id')
    .nunique()
)

# Número de canciones que presentan diferencias
# en cada variable
resumen_variaciones = (
    (variaciones > 1)
    .sum()
    .sort_values(ascending=False)
)

print("Canciones repetidas:", len(ids_repetidos))
print("\nCanciones con diferencias por variable:")

display(resumen_variaciones.to_frame(
    name='Canciones con valores diferentes'
))


Canciones repetidas: 16641

Canciones con diferencias por variable:


,Canciones con valores diferentes
track_genre,16299
popularity,720
artists,0
track_name,0
album_name,0
explicit,0
duration_ms,0
energy,0
key,0
loudness,0


### 1.3.2 Comparación de registros asociados al mismo track_id

Al comparar los registros de una misma canción, se observa que las principales diferencias se concentran en `track_genre` y `popularity`.

Se identifican:

- 16.299 canciones con diferentes géneros.
- 720 canciones con diferentes valores de popularidad.
- Las 720 canciones con popularidad variable también presentan diferentes géneros.
- 15.579 canciones presentan diferencias de género, pero mantienen la misma popularidad.

Los resultados muestran que la repetición de registros está relacionada principalmente con la clasificación musical.

In [80]:

# Canciones cuya popularidad cambia entre registros
ids_popularidad_variable = variaciones[
    variaciones['popularity'] > 1
].index

print(
    "Canciones con diferente popularidad:",
    len(ids_popularidad_variable)
)

# Seleccionamos una canción para investigarla
id_ejemplo = ids_popularidad_variable[0]

columnas = [
    'track_id',
    'track_name',
    'artists',
    'track_genre',
    'popularity',
    'duration_ms',
    'danceability',
    'energy',
    'acousticness',
    'valence',
    'tempo'
]

display(
    df.loc[df['track_id'] == id_ejemplo, columnas]
    .sort_values('track_genre')
)


Canciones con diferente popularidad: 720


,track_id,track_name,artists,track_genre,popularity,duration_ms,danceability,energy,acousticness,valence,tempo
55776,00YwP3wJWiG8IxAA7OS9lo,Amake Amar Moto Thakte Dao,Anupam Roy,indian,47,319946,0.566,0.419,0.771,0.186,147.881
57529,00YwP3wJWiG8IxAA7OS9lo,Amake Amar Moto Thakte Dao,Anupam Roy,indie,47,319946,0.566,0.419,0.771,0.186,147.881
56324,00YwP3wJWiG8IxAA7OS9lo,Amake Amar Moto Thakte Dao,Anupam Roy,indie-pop,47,319946,0.566,0.419,0.771,0.186,147.881
65813,00YwP3wJWiG8IxAA7OS9lo,Amake Amar Moto Thakte Dao,Anupam Roy,k-pop,47,319946,0.566,0.419,0.771,0.186,147.881
99268,00YwP3wJWiG8IxAA7OS9lo,Amake Amar Moto Thakte Dao,Anupam Roy,singer-songwriter,46,319946,0.566,0.419,0.771,0.186,147.881
102269,00YwP3wJWiG8IxAA7OS9lo,Amake Amar Moto Thakte Dao,Anupam Roy,songwriter,46,319946,0.566,0.419,0.771,0.186,147.881



### 1.3.3 Análisis de las diferencias de popularidad

Se observa además que una parte de estas discrepancias se debe a registros con popularidad 0
junto a otros registros de la misma canción con valores positivos (105 canciones).
Estos ceros se consideran errores de registro, ya que una misma canción no puede tener
simultáneamente popularidad nula y positiva.

El tratamiento definitivo de estos casos se define en el apartado 1.3.4.


In [81]:

# Resumir popularidad de cada canción repetida
resumen_pop = (
    df_repetidas
    .groupby('track_id')['popularity']
    .agg(
        apariciones='count',
        valores_distintos='nunique',
        popularidad_min='min',
        popularidad_max='max'
    )
)

# Calcular diferencia entre el máximo y el mínimo
resumen_pop['diferencia'] = (
    resumen_pop['popularidad_max']
    - resumen_pop['popularidad_min']
)

# Canciones cuya popularidad cambia
pop_variable = resumen_pop[
    resumen_pop['valores_distintos'] > 1
]

print("Canciones con popularidad variable:", len(pop_variable))

print("\nDistribución de las diferencias:")
display(pop_variable['diferencia'].value_counts().sort_index())

print("\nEstadísticas de las diferencias:")
display(pop_variable['diferencia'].describe())

print("\nMayores diferencias encontradas:")
display(
    pop_variable.sort_values(
        'diferencia', ascending=False
    ).head(15)
)


Canciones con popularidad variable: 720

Distribución de las diferencias:


diferencia
1     673
2      17
3       7
4       2
5       4
7       2
8       1
10      1
12      1
17      1
18      1
27      1
39      1
40      2
41      3
42      2
44      1
Name: count, dtype: int64


Estadísticas de las diferencias:


count    720.000000
mean       1.711111
std        4.688326
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       44.000000
Name: diferencia, dtype: float64


Mayores diferencias encontradas:


,apariciones,valores_distintos,popularidad_min,popularidad_max,diferencia
track_id,,,,,
5hljqZrl0h7RcIhUnOPuve,3,2,0,44,44
1UwUhKmFxGKs59xiWO60Sx,3,2,0,42,42
0C6dSPCqwKqwos6s1CrDrh,3,2,0,42,42
5ygEUpyZy5qtZ1423zymBW,3,2,0,41,41
1zx6GSqLYI2ynzAHnPRKBR,3,2,0,41,41
4KPi4VNvPOfKqIp1sx19Xf,3,2,0,41,41
5fClRRa7P9nPIpstj50veV,3,2,0,40,40
0z5x3VmmQ7cyTeOP7VAf3T,3,2,0,40,40
42GuKw49pPxNAkIhWGwgFs,3,2,0,39,39


In [82]:

# Comprobar coincidencia entre variaciones de género y popularidad
dif_genero = variaciones['track_genre'] > 1
dif_popularidad = variaciones['popularity'] > 1

print("Con varios géneros:", dif_genero.sum())
print("Con popularidad diferente:", dif_popularidad.sum())

print("Con ambas diferencias:", (dif_genero & dif_popularidad).sum())

print("Solo diferencias de género:", (dif_genero & ~dif_popularidad).sum())

print("Solo diferencias de popularidad:", (~dif_genero & dif_popularidad).sum())


Con varios géneros: 16299
Con popularidad diferente: 720
Con ambas diferencias: 720
Solo diferencias de género: 15579
Solo diferencias de popularidad: 0


In [83]:
# Seleccionar las 10 canciones con mayores diferencias
ids_extremos = (
    pop_variable
    .sort_values('diferencia', ascending=False)
    .head(10)
    .index
)

# Recuperar todos sus registros originales
columnas = [
    'track_id',
    'track_name',
    'artists',
    'album_name',
    'track_genre',
    'popularity'
]

casos_extremos = (
    df.loc[df['track_id'].isin(ids_extremos), columnas]
    .sort_values(['track_id', 'popularity', 'track_genre'])
)

display(casos_extremos)


,track_id,track_name,artists,album_name,track_genre,popularity
83260,0BhLfxWA4TwGu8bGVKBotN,Roll The Dice,Armin van Buuren;Philip Strand,"Feel Again, Pt. 2",progressive-house,0
110105,0BhLfxWA4TwGu8bGVKBotN,Roll The Dice,Armin van Buuren;Philip Strand,"Feel Again, Pt. 2",trance,27
38026,0C6dSPCqwKqwos6s1CrDrh,The Car,Arctic Monkeys,The Car,garage,0
57723,0C6dSPCqwKqwos6s1CrDrh,The Car,Arctic Monkeys,The Car,indie,0
91954,0C6dSPCqwKqwos6s1CrDrh,The Car,Arctic Monkeys,The Car,rock,42
38028,0z5x3VmmQ7cyTeOP7VAf3T,Perfect Sense,Arctic Monkeys,The Car,garage,0
57724,0z5x3VmmQ7cyTeOP7VAf3T,Perfect Sense,Arctic Monkeys,The Car,indie,0
91953,0z5x3VmmQ7cyTeOP7VAf3T,Perfect Sense,Arctic Monkeys,The Car,rock,40
38037,1UwUhKmFxGKs59xiWO60Sx,I Ain’t Quite Where I Think I Am,Arctic Monkeys,The Car,garage,0
57749,1UwUhKmFxGKs59xiWO60Sx,I Ain’t Quite Where I Think I Am,Arctic Monkeys,The Car,indie,0


In [84]:

# Canciones con popularidad mínima 0 y máxima superior a 0
casos_cero_positivo = pop_variable[
    (pop_variable['popularidad_min'] == 0) &
    (pop_variable['popularidad_max'] > 0)
]

print(
    "Canciones con popularidad 0 y otra superior:",
    len(casos_cero_positivo)
)

print(
    "Total de canciones con popularidad variable:",
    len(pop_variable)
)

print(
    "Porcentaje:",
    round(
        len(casos_cero_positivo) / len(pop_variable) * 100, 2
    ),
    "%"
)


Canciones con popularidad 0 y otra superior: 105
Total de canciones con popularidad variable: 720
Porcentaje: 14.58 %



### 1.3.4 Criterios para el tratamiento de la popularidad

Se ha comprobado que las diferencias de popularidad suelen ser pequeñas.

De las 615 canciones analizadas, 613 (99,67 %) presentan diferencias de tres puntos o menos. Únicamente dos canciones superan este umbral, con diferencias de 7 y 12 puntos.

Dado que no disponemos de información temporal que permita determinar cuál es el valor correcto, se excluiran estas dos canciones del análisis de popularidad y del modelo predictivo.

Para el resto de canciones con diferentes valores positivos se utilizará la mediana como valor representativo. Cuando existan valores cero junto con valores positivos para un mismo identificador, se excluirán los ceros en la consolidación.

In [85]:

# Excluir temporalmente las canciones que mezclan
# popularidad cero con popularidad positiva

ids_con_cero = casos_cero_positivo.index

pop_variable_sin_ceros = pop_variable.loc[
    ~pop_variable.index.isin(ids_con_cero)
].copy()

print(
    "Canciones restantes:",
    len(pop_variable_sin_ceros)
)

print("\nDistribución de las diferencias:")
display(
    pop_variable_sin_ceros['diferencia']
    .value_counts()
    .sort_index()
)

print("\nEstadísticas descriptivas:")
display(
    pop_variable_sin_ceros['diferencia'].describe()
)

print(
    "\nCanciones con diferencias superiores a 5 puntos:",
    (pop_variable_sin_ceros['diferencia'] > 5).sum()
)

print("\nMayores diferencias:")
display(
    pop_variable_sin_ceros
    .sort_values('diferencia', ascending=False)
    .head(15)
)


Canciones restantes: 615

Distribución de las diferencias:


diferencia
1     602
2       8
3       3
7       1
12      1
Name: count, dtype: int64


Estadísticas descriptivas:


count    615.000000
mean       1.050407
std        0.534536
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       12.000000
Name: diferencia, dtype: float64


Canciones con diferencias superiores a 5 puntos: 2

Mayores diferencias:


,apariciones,valores_distintos,popularidad_min,popularidad_max,diferencia
track_id,,,,,
6mPet122yXpp6X8iTiNVo9,6,2,4,16,12
2s3Hr38fRfDRj9JWVP494L,2,2,7,14,7
0v8tmDMQrTMavluubqAJo4,5,2,1,4,3
7A7FsIDnBMnz1PAnSOxUdh,3,2,58,61,3
7FjBkz3OJiknpx1jtNkIFs,2,2,4,7,3
0vgIkHFK8wpogVc1XCi18F,2,2,39,41,2
1EWRXz6c0WPKae8vziEnZi,3,2,1,3,2
2DOdnZIgqRPq95tDc26Kgm,2,2,17,19,2
3DwkoCj5vhaKzgdbEpoDIu,2,2,2,4,2



## 1.4 Inspección de valores nulos

Se detectaron tres valores nulos, correspondientes a las variables `artists`, `album_name` y `track_name`.

Hemos visto que son de la misma canción por lo que la eliminaremos


In [86]:

# Número de valores nulos por columna
nulos = df.isnull().sum()

print("Valores nulos por variable:")
display(nulos[nulos > 0].to_frame(name='Valores nulos'))

# Identificar las filas afectadas
filas_nulas = df[
    df.isnull().any(axis=1)
]

print("Número de filas afectadas:", len(filas_nulas))

# Mostrar los registros afectados
display(filas_nulas.T)


Valores nulos por variable:


,Valores nulos
artists,1
album_name,1
track_name,1


Número de filas afectadas: 1


,65900
track_id,1kR4gIb7nGxHPI3D2ifs59
artists,NaN
album_name,NaN
track_name,NaN
popularity,0
duration_ms,0
explicit,False
danceability,0.501
energy,0.583
key,7


## 1.5 Detección de valores anómalos

Además de nulos y duplicados, revisamos si existen valores imposibles o poco realistas
en las variables numéricas: `tempo`, `time_signature` y `duration_ms`.

In [87]:
# Trabajamos con canciones únicas para no contar varias veces la misma
df_unicas = df.drop_duplicates(subset='track_id')

# Tempo igual a 0
tempo_cero = df_unicas[df_unicas['tempo'] == 0]
print("Canciones con tempo = 0:", len(tempo_cero))
print("De ellas, con time_signature = 0:", (tempo_cero['time_signature'] == 0).sum())
print("\nGéneros más frecuentes con tempo 0:")
display(tempo_cero['track_genre'].value_counts().head(5))

# Distribución del compás y su relación con la popularidad
print("Distribución de time_signature (%):")
display((df_unicas['time_signature'].value_counts(normalize=True) * 100).round(2))

print("Popularidad media por compás:")
display(df_unicas.groupby('time_signature')['popularity'].agg(['count', 'mean']).round(1))

Canciones con tempo = 0: 157
De ellas, con time_signature = 0: 157

Géneros más frecuentes con tempo 0:


track_genre
sleep          138
guitar           4
iranian          4
ambient          3
world-music      3
Name: count, dtype: int64

Distribución de time_signature (%):


time_signature
4    88.64
3     8.47
5     1.77
1     0.94
0     0.18
Name: proportion, dtype: float64

Popularidad media por compás:


,count,mean
time_signature,,
0,162,36.9
1,846,28.1
3,7604,29.6
4,79544,33.7
5,1585,28.8


In [88]:
duracion_min = df_unicas['duration_ms'] / 60000

print("Duración mínima (ms):", df_unicas['duration_ms'].min())
print("Duración máxima (min):", round(duracion_min.max(), 1))
print("Canciones de más de 15 minutos:", (duracion_min > 15).sum())
print("Canciones de menos de 30 segundos:", (duracion_min < 0.5).sum())

# Ver las más largas
display(
    df_unicas.loc[duracion_min.sort_values(ascending=False).head(10).index,
                  ['track_name', 'artists', 'track_genre', 'duration_ms']]
)

Duración mínima (ms): 0
Duración máxima (min): 87.3
Canciones de más de 15 minutos: 141
Canciones de menos de 30 segundos: 15


,track_name,artists,track_genre,duration_ms
73617,Unity (Voyage Mix) Pt. 1,Tale Of Us,minimal-techno,5237295
10935,Crossing Wires 002 - Continuous DJ Mix,Timo Maas,breakbeat,4789026
24348,The Lab 03 - Continuous DJ Mix Part 1,Seth Troxler,detroit-techno,4730302
73840,Amnesia Ibiza Underground 10 DJ Mix,Loco Dice,minimal-techno,4563897
13344,House of Om - Mark Farina - Continuous Mix,Mark Farina,chicago-house,4447520
13245,Live In Tokyo - Continuous Mix,Mark Farina,chicago-house,4339826
13195,Greenhouse Construction,Mark Farina,chicago-house,4334721
27926,"NQ State of Mind, Vol. 1 - Continuous DJ Mix",Lenzman;Dan Stezo,drum-and-bass,4246206
101390,Ocean Waves Sounds,Ocean Sounds,sleep,4120258
45900,Internal Flight,Estas Tonne,guitar,3876276


#### Conclusiones

- **Tempo:** 157 canciones tienen `tempo = 0`. Todas tienen también `time_signature = 0`
  y la mayoría pertenecen al género *sleep* (138). Son pistas de ruido blanco o ambientales
  en las que Spotify no detecta ritmo, por lo que no representan canciones musicales.
- **time_signature:** el 88,64 % de las canciones está en 4/4, por lo que la variable aporta
  muy poca información. Además contiene valores no válidos (0 y 1) y la popularidad media
  apenas varía entre compases. **Se decide eliminar esta variable.**
- **Duración:** la canción de 0 ms coincide con el registro con nulos (se eliminará).
  Hay 141 canciones de más de 15 minutos y 15 de menos de 30 segundos; no son errores
  necesariamente, pero se tendrán en cuenta como valores atípicos.

# Resumen de decisiones para la limpieza

| Problema detectado | Decisión |
|---|---|
| 450 filas duplicadas exactas | Eliminar |
| 1 fila con nulos (artists, album_name, track_name) | Eliminar |
| Canciones repetidas por varios géneros | Consolidar en una fila con `track_genres` |
| Popularidad 0 junto a valores positivos | Ignorar los ceros |
| Diferencias de popularidad ≤ 5 puntos | Usar la mediana |
| Diferencias de popularidad > 5 puntos | Excluir la canción |
| 157 canciones con tempo = 0 | Decidir: eliminar (recomendado) |
| `time_signature` | Eliminar la variable |
| Duraciones extremas | Mantener y tratar como atípicos |